In [12]:
import random
import datetime
from faker import Faker
import psycopg2

In [18]:
fake = Faker()
NUM_USERS = 50
NUM_ANONS = 50
ACTIONS_PER_DAY = 5
DAYS = 30

DB_PARAMS = {
    "dbname": "forumdb",
    "user": "user",
    "password": "password",
    "host": "localhost",
    "port": 5432,
}

# Получение ID действия по названию
ACTION_TYPE_IDS = {
    "first_visit": 1,
    "registration": 2,
    "login": 3,
    "logout": 4,
    "topic_create": 5,
    "topic_view": 6,
    "topic_delete": 7,
    "message_post": 8,
}

def connect_db():
    return psycopg2.connect(**DB_PARAMS)

def create_fake_users(conn):
    user_ids = []
    with conn.cursor() as cur:
        for _ in range(NUM_USERS):
            username = fake.user_name()
            email = fake.email()
            password = fake.sha256()
            cur.execute("""
                INSERT INTO users (username, email, password_hash, registration_date)
                VALUES (%s, %s, %s, %s) RETURNING user_id
            """, (username, email, password, fake.date_time_this_year()))
            user_ids.append(cur.fetchone()[0])
        conn.commit()
    return user_ids

def create_fake_anons(conn):
    anon_ids = []
    with conn.cursor() as cur:
        for _ in range(NUM_ANONS):
            session_id = fake.uuid4()
            ip = fake.ipv4()
            user_agent = fake.user_agent()
            cur.execute("""
                INSERT INTO anonymous_users (session_id, ip_address, user_agent, first_seen)
                VALUES (%s, %s, %s, %s) RETURNING anon_id
            """, (session_id, ip, user_agent, fake.date_time_this_year()))
            anon_ids.append(cur.fetchone()[0])
        conn.commit()
    return anon_ids

def simulate_logs(conn, user_ids, anon_ids):
    start_date = datetime.datetime.now() - datetime.timedelta(days=DAYS)
    topic_ids = []

    for day_offset in range(DAYS):
        day = start_date + datetime.timedelta(days=day_offset)
        logs_today = []

        for action, action_id in ACTION_TYPE_IDS.items():
            min_count = 5
            max_count = min_count + random.randint(0, 5)
            for _ in range(max_count):
                ts = day + datetime.timedelta(minutes=random.randint(0, 1440))

                # === Определяем user/anon
                user_id, anon_id = None, None
                if action == "topic_create":
                    if random.random() < 0.8:
                        user_id = random.choice(user_ids)
                        response = "success"
                        with conn.cursor() as cur:
                            cur.execute("""
                                INSERT INTO topics (title, content, created_by, created_at)
                                VALUES (%s, %s, %s, %s) RETURNING topic_id
                            """, (fake.sentence(), fake.text(), user_id, ts))
                            topic_id = cur.fetchone()[0]
                            topic_ids.append(topic_id)
                    else:
                        response = "error"
                        topic_id = None
                elif action == "message_post":
                    if random.random() < 0.5:
                        user_id = random.choice(user_ids)
                    else:
                        anon_id = random.choice(anon_ids)

                    if not topic_ids:
                        continue  # нет тем — пропускаем

                    topic_id = random.choice(topic_ids)
                    with conn.cursor() as cur:
                        cur.execute("""
                            INSERT INTO messages (topic_id, content, created_at, user_id, anon_id)
                            VALUES (%s, %s, %s, %s, %s) RETURNING message_id
                        """, (
                            topic_id, fake.text(), ts, user_id, anon_id
                        ))
                        msg_id = cur.fetchone()[0]
                    response = "success"
                else:
                    if random.random() < 0.7:
                        user_id = random.choice(user_ids)
                    else:
                        anon_id = random.choice(anon_ids)
                    topic_id = random.choice(topic_ids) if topic_ids else None
                    response = "success"

                with conn.cursor() as cur:
                    cur.execute("""
                        INSERT INTO user_logs (
                            action_type_id, user_id, anon_id, action_time, entity_type, entity_id,
                            server_response, additional_info, ip_address, user_agent
                        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        action_id, user_id, anon_id, ts,
                        'topic' if action.startswith('topic') else 'message' if action == 'message_post' else None,
                        topic_id,
                        response, fake.text(max_nb_chars=50),
                        fake.ipv4(), fake.user_agent()
                    ))
        conn.commit()

def main():
    conn = connect_db()
    user_ids = create_fake_users(conn)
    anon_ids = create_fake_anons(conn)
    simulate_logs(conn, user_ids, anon_ids)
    conn.close()
    print("Логи успешно сгенерированы.")

if __name__ == "__main__":
    main()


UndefinedTable: relation "users" does not exist
LINE 2:                 INSERT INTO users (username, email, password...
                                    ^
